In [23]:
# import the necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import roc_auc_score


In [ ]:
# load the dataset into pandas dataframe
path = '../data/features/loan_features.csv'
df = pd.read_csv(path)
df.head()

,loan_amnt,term,int_rate,installment,purpose,application_type,disbursement_method,initial_list_status,emp_length,home_ownership,...,recent_inquiry_flag,inquiry_rate,credit_age_bucket,credit_breadth,thin_file_flag,income_segment,income_verified_flag,revol_utilization_ratio,avg_credit_limit_per_account,pct_maxed_cards
0,35000.0,36,12.12,1164.51,credit_card,Individual,Cash,f,5.0,RENT,...,0,0.000000,Seasoned,0.625388,0,Formal_High,1,0.165333,166.666667,0.000
1,5000.0,36,8.99,158.98,debt_consolidation,Individual,Cash,f,10.0,RENT,...,1,0.000000,Seasoned,0.759976,0,Informal,0,0.326026,278.571429,0.000
2,23000.0,36,7.89,719.57,debt_consolidation,Individual,Cash,w,10.0,MORTGAGE,...,0,0.000000,Seasoned,1.056060,0,Formal_Mid,1,0.448632,852.777778,0.000
3,12000.0,36,6.97,370.37,credit_card,Individual,Cash,w,1.0,MORTGAGE,...,1,0.022553,Seasoned,0.518725,0,Informal,0,0.482419,2408.695652,0.167
4,15000.0,36,7.69,467.91,debt_consolidation,Individual,Cash,f,10.0,OWN,...,1,0.036463,Established,1.239742,0,Informal,0,0.502846,1085.294118,0.500


In [3]:
# perform a quick overview of the dataset
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 57722 entries, 0 to 57721
Data columns (total 62 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   loan_amnt                     57722 non-null  float64
 1   term                          57722 non-null  int64  
 2   int_rate                      57722 non-null  float64
 3   installment                   57722 non-null  float64
 4   purpose                       57722 non-null  str    
 5   application_type              57722 non-null  str    
 6   disbursement_method           57722 non-null  str    
 7   initial_list_status           57722 non-null  str    
 8   emp_length                    54332 non-null  float64
 9   home_ownership                57722 non-null  str    
 10  annual_inc                    57721 non-null  float64
 11  verification_status           57722 non-null  str    
 12  addr_state                    57722 non-null  str    
 13  dti         

In [12]:
# create a list for the numeric and the categorical variables

numerical = list(df.select_dtypes(include=['int64', 'float64']).columns)
categorical = list(df.select_dtypes(include=['str']).columns)

In [21]:
df[numerical].isnull().sum()

loan_amnt                          0
term                               0
int_rate                           0
installment                        0
emp_length                      3390
annual_inc                         1
dti                               18
delinq_2yrs                        2
inq_last_6mths                     2
open_acc                           2
pub_rec                            2
revol_bal                          0
revol_util                        35
total_acc                          2
pub_rec_bankruptcies              52
tax_liens                          3
collections_12_mths_ex_med         5
acc_now_delinq                     2
chargeoff_within_12_mths           5
delinq_amnt                        2
total_rev_hi_lim                3132
bc_util                         2822
percent_bc_gt_75                2818
avg_cur_bal                     3132
total_bal_ex_mort               2209
total_bc_limit                  2209
tot_cur_bal                     3132
t

In [ ]:
def build_pipeline(model, path):
    # load the dataset into pandas dataframe

    print("Loading features...")
    df = pd.read_csv(path)

    numerical = list(df.select_dtypes(include=['int64', 'float64']).columns)
    categorical = list(df.select_dtypes(include=['str']).columns)

    categorical_preprocessor = Pipeline(steps=[
        ("cat", OneHotEncoder(drop='first', sparse_output=False))
    ])

    numerical_preprocessor = Pipeline(steps=[
        ("numeric", SimpleImputer(strategy='median')),
        ("scaler", StandardScaler())
    ]) 

    preprocessor = ColumnTransformer(transformers=[
        ('cat', categorical_preprocessor, categorical),
        ('num', numerical_preprocessor, numerical)
    ])

    # Full ML pipeline
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    X = df[numerical+categorical]
    X = X.drop(['default_flag'], axis=1)

    y = df['default_flag']

    # split data into train and test set
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Train & evaluate
    print("Training model...")
    pipeline.fit(X_train, y_train)

    # Predictions
    y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
    y_pred = pipeline.predict(X_test)

    #evaluate
    

    

